# Laboratório – Instruction Tuning com GPT-2 e Avaliação com LLM-as-a-Judge

**Objetivos:** Construir dataset de instruções (AG News + SST-2), fine-tune GPT-2-medium, avaliar com LLM-as-a-Judge.

**Pipeline:** Dataset → Variações de instrução → Curadoria → Split 80/10/10 → Fine-tuning → Avaliação

In [20]:
from importlib.metadata import version
import json
import os
import random
import requests
import torch
from torch.utils.data import Dataset
import tiktoken
from torch.utils.data import DataLoader
from tqdm import tqdm
import re
import psutil
from datasets import load_dataset
from pathlib import Path
from typing import List, Dict, Optional
from functools import partial

pkgs = ["numpy", "matplotlib", "tiktoken", "torch", "tqdm", "transformers", "datasets"]
for p in pkgs:
    try:
        print(f"{p} version: {version(p)}")
    except Exception:
        print(f"{p}: (not installed)")


numpy version: 2.0.2
matplotlib version: 3.10.0
tiktoken version: 0.12.0
torch version: 2.9.0+cpu
tqdm version: 4.67.3
transformers version: 5.0.0
datasets version: 4.0.0


# Dataset

In [21]:
def load_ag_news(max_samples: Optional[int] = None) -> List[Dict]:
    ds = load_dataset("ag_news", split="train", trust_remote_code=True)
    labels = ["World", "Sports", "Business", "Sci/Tech"]

    data = []
    for i, ex in enumerate(ds):
        if max_samples and i >= max_samples:
            break
        data.append({
            "text": ex["text"],
            "label": labels[ex["label"]],
            "label_id": ex["label"]
        })
    return data


def load_sst2(max_samples: Optional[int] = None) -> List[Dict]:
    ds = load_dataset("glue", "sst2", split="train", trust_remote_code=True)
    labels = ["negative", "positive"]

    data = []
    for i, ex in enumerate(ds):
        if max_samples and i >= max_samples:
            break
        data.append({
            "text": ex["sentence"],
            "label": labels[ex["label"]],
            "label_id": ex["label"]
        })
    return data


def convert_classification_to_instruction_format(
    data: List[Dict],
    task: str = "ag_news"
) -> List[Dict]:
    """Converte dataset de classificação para formato instrução/entrada/resposta."""
    if task == "ag_news":
        labels = "World, Sports, Business, Sci/Tech"
        instruction = f"Classifique o texto nas categorias: {labels}."
    elif task == "sst2":
        labels = "negative, positive"
        instruction = f"Classifique o sentimento do texto nas categorias: {labels}."
    else:
        labels = "classe1, classe2"
        instruction = f"Classifique o texto nas categorias: {labels}."

    result = []
    for item in data:
        result.append({
            "instruction": instruction,
            "input": item["text"],
            "output": item["label"],
            "source": task
        })
    return result


def load_and_combine_ag_news_sst2(
    max_per_dataset: Optional[int] = None
) -> List[Dict]:
    """
    Carrega AG News e SST-2, converte para instruções e combina.
    max_per_dataset: limite por dataset (ex: 1000 cada = 2000 total).
    """
    ag_raw = load_ag_news(max_samples=max_per_dataset)
    sst_raw = load_sst2(max_samples=max_per_dataset)

    ag_data = convert_classification_to_instruction_format(ag_raw, task="ag_news")
    sst_data = convert_classification_to_instruction_format(sst_raw, task="sst2")

    combined = ag_data + sst_data
    random.shuffle(combined)
    return combined



# Instruções LLM

In [22]:
INSTRUCTION_GENERATION_PROMPT = """
Dado o seguinte exemplo de tarefa de NLP, gere 3 variações diferentes e realistas
da instrução que um usuário poderia dar. Mantenha o mesmo objetivo da tarefa.
Retorne APENAS as 3 instruções, uma por linha, sem numeração.

Exemplo original: {instruction}
Contexto: classificação de texto
"""


def generate_instruction_variations_with_llm(
    data: List[Dict],
    api_func,
    sample_ratio: float = 0.3,
    seed: int = 42
) -> List[Dict]:
    """
    Usa LLM para gerar variações de instruções em uma amostra.
    api_func: callable que recebe prompt e retorna texto gerado.
    """
    random.seed(seed)
    n_sample = max(1, int(len(data) * sample_ratio))
    indices = random.sample(range(len(data)), min(n_sample, len(data)))

    instruction_pool = {}
    for idx in tqdm(indices, desc="Gerando variações de instrução"):
        entry = data[idx]
        inst = entry["instruction"]
        if inst not in instruction_pool:
            prompt = INSTRUCTION_GENERATION_PROMPT.format(instruction=inst)
            try:
                variations = api_func(prompt).strip().split("\n")
                variations = [v.strip().lstrip("0123456789.-) ") for v in variations if v.strip()]
                instruction_pool[inst] = variations[:3] if variations else [inst]
            except Exception as e:
                instruction_pool[inst] = [inst]

    # Aplica variações na amostra
    result = []
    for i, entry in enumerate(data):
        new_entry = entry.copy()
        if i in indices and entry["instruction"] in instruction_pool:
            pool = instruction_pool[entry["instruction"]]
            new_entry["instruction"] = random.choice(pool) if pool else entry["instruction"]
        result.append(new_entry)

    return result


def generate_instruction_variations_static(data: List[Dict]) -> List[Dict]:
    """
    Alternativa SEM API: usa variações pré-definidas (não precisa de LLM externo).
    """
    variations = {
        "Classifique o texto nas categorias: World, Sports, Business, Sci/Tech.": [
            "Qual a categoria desta notícia? Opções: World, Sports, Business, Sci/Tech.",
            "Atribua uma das categorias ao texto: World, Sports, Business ou Sci/Tech.",
            "Identifique se o texto é sobre World, Sports, Business ou Sci/Tech.",
        ],
        "Classifique o sentimento do texto nas categorias: negative, positive.": [
            "O texto é positivo ou negativo?",
            "Qual o sentimento expresso: negative ou positive?",
            "Identifique a polaridade do texto: negative ou positive.",
        ]
    }

    result = []
    for entry in data:
        inst = entry["instruction"]
        if inst in variations:
            new_entry = entry.copy()
            new_entry["instruction"] = random.choice(variations[inst])
            result.append(new_entry)
        else:
            result.append(entry.copy())

    random.shuffle(result)
    return result



# Filtragem

In [23]:
def filter_vague_instructions(data: List[Dict]) -> List[Dict]:
    """Remove instruções vagas (muito curtas ou genéricas)."""
    vague_patterns = [
        r"^[Ff]aça\s+algo\.?$",
        r"^[Rr]esponda\.?$",
        r"^[Aa]nalise\.?$",
        r"^.{1,15}$",  # instrução com menos de 15 chars
    ]
    compiled = [re.compile(p) for p in vague_patterns]

    filtered = []
    for entry in data:
        inst = entry.get("instruction", "")
        if any(c.search(inst) for c in compiled):
            continue
        filtered.append(entry)
    return filtered


def filter_label_leakage(data: List[Dict], tasks: List[str] = None) -> List[Dict]:
    """Remove exemplos onde o rótulo aparece no input de forma suspeita (vazamento)."""
    tasks = tasks or ["ag_news", "sst2"]
    filtered = []
    for entry in data:
        output = entry.get("output", "").strip()
        input_text = entry.get("input", "").lower()
        if output.lower() in input_text and len(output) > 3:
            continue
        filtered.append(entry)
    return filtered


def find_duplicate_indices(data: List[Dict], key: str = "input", threshold: float = 0.95) -> set:
    """
    Encontra índices de duplicatas/near-duplicatas por similaridade de texto.
    Usa Jaccard em tokens como aproximação (sem sklearn).
    """
    def tokenize(s: str) -> set:
        return set(re.findall(r"\w+", s.lower()))

    def jaccard(a: set, b: set) -> float:
        if not a and not b:
            return 1.0
        inter = len(a & b)
        union = len(a | b)
        return inter / union if union else 0.0

    to_remove = set()
    texts = [(i, tokenize(entry.get(key, ""))) for i, entry in enumerate(data)]

    for i in range(len(texts)):
        if i in to_remove:
            continue
        for j in range(i + 1, len(texts)):
            if j in to_remove:
                continue
            sim = jaccard(texts[i][1], texts[j][1])
            if sim >= threshold:
                to_remove.add(j)

    return to_remove


def filter_duplicates(data: List[Dict], keys: List[str] = None, threshold: float = 0.9) -> List[Dict]:
    """Remove duplicatas e near-duplicatas."""
    keys = keys or ["instruction", "input"]
    all_to_remove = set()

    for key in keys:
        to_remove = find_duplicate_indices(data, key=key, threshold=threshold)
        all_to_remove.update(to_remove)

    return [entry for i, entry in enumerate(data) if i not in all_to_remove]


VALID_OUTPUTS_AG_NEWS = {"World", "Sports", "Business", "Sci/Tech"}
VALID_OUTPUTS_SST2 = {"negative", "positive"}
VALID_OUTPUTS_COMBINED = VALID_OUTPUTS_AG_NEWS | VALID_OUTPUTS_SST2


def filter_inconsistent(data: List[Dict], tasks: List[str] = None) -> List[Dict]:
    """Remove exemplos inconsistentes (output não está nas opções válidas)."""
    tasks = tasks or ["ag_news", "sst2"]
    valid = VALID_OUTPUTS_COMBINED
    return [e for e in data if e.get("output", "").strip() in valid]


def apply_curation_pipeline(data: List[Dict], tasks: List[str] = None) -> List[Dict]:
    """Aplica toda a curadoria e filtragem. tasks: ["ag_news", "sst2"]."""
    tasks = tasks or ["ag_news", "sst2"]
    data = filter_vague_instructions(data)
    data = filter_label_leakage(data, tasks)
    data = filter_inconsistent(data, tasks)
    data = filter_duplicates(data, threshold=0.92)
    return data



# Normalização

In [24]:
def normalize_entry(entry: Dict) -> Dict:
    """Padroniza formato: instruction, input, output (strings limpas)."""
    out = {
        "instruction": str(entry.get("instruction", "")).strip(),
        "input": str(entry.get("input", "")).strip(),
        "output": str(entry.get("output", "")).strip()
    }
    if "source" in entry:
        out["source"] = entry["source"]
    return out


def normalize_dataset(data: List[Dict]) -> List[Dict]:
    """Normaliza todo o dataset para formato padrão."""
    normalized = []
    for entry in data:
        n = normalize_entry(entry)
        if n["instruction"] and n["output"]:
            normalized.append(n)
    return normalized


def to_alpaca_format(entry: Dict) -> str:
    """Converte para formato Alpaca (texto linear para treino)."""
    parts = [
        "Below is an instruction that describes a task. Write a response that appropriately completes the request.",
        "",
        "### Instruction:",
        entry["instruction"],
        "",
        "### Input:",
        entry["input"] if entry["input"] else "(empty)",
        "",
        "### Response:",
        entry["output"]
    ]
    return "\n".join(parts)

# Split

In [25]:
def split_dataset(
    data: List[Dict],
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42
) -> tuple:
    """
    Split: 80% treino, 10% validação, 10% teste.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    random.seed(seed)
    data = data.copy()
    random.shuffle(data)

    n = len(data)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    n_test = n - n_train - n_val

    train_data = data[:n_train]
    val_data = data[n_train:n_train + n_val]
    test_data = data[n_train + n_val:]

    return train_data, val_data, test_data


In [26]:
def main():
    OUTPUT_DIR = Path(".") / "data"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    MAX_PER_DATASET = 1000  # 1000 AG News + 1000 SST-2 = 2000 total

    print("=" * 60)
    print("SEÇÃO 1: Carregando datasets (AG News + SST-2)")
    print("=" * 60)
    data = load_and_combine_ag_news_sst2(max_per_dataset=MAX_PER_DATASET)
    n_ag = sum(1 for e in data if e.get("source") == "ag_news")
    n_sst = sum(1 for e in data if e.get("source") == "sst2")
    print(f"AG News: {n_ag} | SST-2: {n_sst} | Total: {len(data)}")

    print("\n" + "=" * 60)
    print("SEÇÃO 2: Variações de instrução (estático, sem API)")
    print("=" * 60)
    data = generate_instruction_variations_static(data)
    print("Exemplos após variação (AG News e SST-2):")
    for e in data[:2]:
        print(json.dumps(e, indent=2, ensure_ascii=False))
        print("---")

    print("\n" + "=" * 60)
    print("SEÇÃO 3: Curadoria e filtragem")
    print("=" * 60)
    before = len(data)
    data = apply_curation_pipeline(data, tasks=["ag_news", "sst2"])
    print(f"Antes: {before} | Depois: {len(data)}")

    print("\n" + "=" * 60)
    print("SEÇÃO 4: Normalização")
    print("=" * 60)
    data = normalize_dataset(data)
    print(f"Exemplos normalizados: {len(data)}")

    print("\n" + "=" * 60)
    print("SEÇÃO 5: Split (80/10/10)")
    print("=" * 60)
    train, val, test = split_dataset(data, seed=42)
    print(f"Treino: {len(train)} | Val: {len(val)} | Teste: {len(test)}")

    # Salvar
    for name, subset in [("train", train), ("val", val), ("test", test)]:
        path = OUTPUT_DIR / f"instruction_{name}.jsonl"
        with open(path, "w", encoding="utf-8") as f:
            for entry in subset:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        print(f"Salvo: {path}")

    full_path = OUTPUT_DIR / "instruction_dataset.json"
    with open(full_path, "w", encoding="utf-8") as f:
        json.dump({"train": train, "val": val, "test": test}, f, indent=2, ensure_ascii=False)
    print(f"Salvo: {full_path}")

    print("\nPipeline concluído.")
    return train, val, test


In [27]:
# Executar pipeline de construção do dataset
train_data, val_data, test_data = main()

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ag_news' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ag_news' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


SEÇÃO 1: Carregando datasets (AG News + SST-2)


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'glue' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


AG News: 1000 | SST-2: 1000 | Total: 2000

SEÇÃO 2: Variações de instrução (estático, sem API)
Exemplos após variação (AG News e SST-2):
{
  "instruction": "Qual a categoria desta notícia? Opções: World, Sports, Business, Sci/Tech.",
  "input": "Playing the convergence game Sony and Microsoft confront consumer apathy as they attempt to turn game consoles into multipurpose entertainment gadgets.",
  "output": "Sci/Tech",
  "source": "ag_news"
}
---
{
  "instruction": "Identifique a polaridade do texto: negative ou positive.",
  "input": "manages to accomplish what few sequels can -- it equals the original and in some ways even betters it ",
  "output": "positive",
  "source": "sst2"
}
---

SEÇÃO 3: Curadoria e filtragem
Antes: 2000 | Depois: 6

SEÇÃO 4: Normalização
Exemplos normalizados: 6

SEÇÃO 5: Split (80/10/10)
Treino: 4 | Val: 0 | Teste: 2
Salvo: data/instruction_train.jsonl
Salvo: data/instruction_val.jsonl
Salvo: data/instruction_test.jsonl
Salvo: data/instruction_dataset.json


# format_input e InstructionDataset (ch07 - Alpaca)

Formato Alpaca: Instrução + Entrada + Resposta. O modelo aprende a completar a seção Response dado o prompt.

In [28]:
def format_input(entry: Dict) -> str:
    """Formato Alpaca: instrução + input (sem resposta). Usado para inferência."""
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry.get("input") else ""
    return instruction_text + input_text


class InstructionDataset(Dataset):
    """Dataset que pré-tokeniza textos no formato Alpaca (ch07)."""
    def __init__(self, data: List[Dict], tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            ids = tokenizer.encode(full_text, add_special_tokens=False)
            self.encoded_texts.append(ids)

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [29]:
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    """Agrupa batch com padding, targets shift+1, e ignore_index para padding (ch07)."""
    batch_max_length = max(len(item) + 1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = list(item) + [pad_token_id]
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    return torch.stack(inputs_lst).to(device), torch.stack(targets_lst).to(device)

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

from transformers import GPT2Tokenizer, GPT2LMHeadModel
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
tokenizer.pad_token = tokenizer.eos_token  # 50256

Device: cpu


In [31]:
BATCH_SIZE = 8
customized_collate = partial(custom_collate_fn, device=device, allowed_max_length=1024)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=customized_collate, shuffle=True, drop_last=True)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=customized_collate, shuffle=False, drop_last=False)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=customized_collate, shuffle=False, drop_last=False)
print(f"Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

Train batches: 0, Val: 0, Test: 1


# Fine-tuning GPT-2-medium (código base Rashka, implementação com transformers)

Hiperparâmetros (ch07): LR=5e-5, weight_decay=0.1, 2 épocas

In [32]:
model = GPT2LMHeadModel.from_pretrained("gpt2-medium")
model.to(device)

def calc_loss_loader(loader, model, device, num_batches=None):
    model.eval()
    total_loss = 0
    n = 0
    with torch.no_grad():
        for i, (inputs, targets) in enumerate(loader):
            if num_batches and i >= num_batches:
                break
            logits = model(inputs).logits
            loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
            total_loss += loss.item()
            n += 1
    return total_loss / n if n else 0

with torch.no_grad():
    train_loss_init = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss_init = calc_loss_loader(val_loader, model, device, num_batches=5)
print(f"Train loss inicial: {train_loss_init:.4f}, Val loss inicial: {val_loss_init:.4f}")

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Train loss inicial: 0.0000, Val loss inicial: 0.0000


In [33]:
import time
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
NUM_EPOCHS = 2
eval_freq = 5

train_losses, val_losses = [], []
torch.manual_seed(123)
start = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    for step, (inputs, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        logits = model(inputs).logits
        loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
        if step % eval_freq == 0:
            val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
            val_losses.append(val_loss)
            print(f"Ep {epoch+1} (Step {step:05d}): Train loss {loss.item():.3f}, Val loss {val_loss:.3f}")

print(f"Treino concluído em {(time.time()-start)/60:.2f} min")

Treino concluído em 0.00 min


In [34]:
torch.save(model.state_dict(), "gpt2-medium-sft.pth")
print("Modelo salvo: gpt2-medium-sft.pth")

Modelo salvo: gpt2-medium-sft.pth


# Geração de respostas: GPT-2 base vs fine-tuned

Para cada exemplo de teste: gerar com modelo base e com fine-tuned.

In [35]:
def generate_response(model, entry, tokenizer, max_new_tokens=50, temperature=0.7):
    """Gera resposta dado instruction+input."""
    prompt = format_input(entry)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id, temperature=temperature, do_sample=True)
    full = tokenizer.decode(out[0], skip_special_tokens=False)
    response = full[len(prompt):].replace("### Response:", "").strip()
    if "<|endoftext|>" in response:
        response = response.split("<|endoftext|>")[0].strip()
    return response

model_base = GPT2LMHeadModel.from_pretrained("gpt2-medium")
model_base.to(device)
model_base.eval()

N_TEST = min(20, len(test_data))
torch.manual_seed(42)
test_with_responses = []
for i in range(N_TEST):
    entry = test_data[i].copy()
    entry["response_base"] = generate_response(model_base, entry, tokenizer)
    entry["response_finetuned"] = generate_response(model, entry, tokenizer)
    test_with_responses.append(entry)
    if (i+1) % 5 == 0:
        print(f"Geradas {i+1}/{N_TEST} respostas")

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# LLM-as-a-Judge

Para cada exemplo: enviar instrução, entrada e as duas respostas (A=base, B=fine-tuned) para LLM juíza.
Scores: correção factual (0-5), aderência à instrução (0-5), clareza/utilidade (0-5).
Vencedor: A, B ou empate + justificativa.

In [36]:
def query_ollama(prompt: str, model: str = "llama3", url: str = "http://localhost:11434/api/chat") -> str:
    """Chama Ollama (Llama 3) como juiz. Requer ollama serve rodando."""
    data = {"model": model, "messages": [{"role": "user", "content": prompt}], "options": {"seed": 123, "temperature": 0}}
    try:
        with requests.post(url, json=data, stream=True, timeout=60) as r:
            r.raise_for_status()
            out = ""
            for line in r.iter_lines(decode_unicode=True):
                if line:
                    j = json.loads(line)
                    if "message" in j:
                        out += j["message"]["content"]
            return out
    except Exception as e:
        return f"[ERRO Ollama: {e}. Instale ollama e rode 'ollama serve' + 'ollama run llama3']"

JUDGE_PROMPT = """Você é um juiz imparcial. Avalie as duas respostas (A e B) para a mesma instrução e entrada.

Instrução: {instruction}
Entrada: {input_text}

Resposta A (modelo base): {response_a}
Resposta B (modelo fine-tuned): {response_b}

Resposta esperada (referência): {reference}

Forneça:
1. Correção factual (0-5): A e B
2. Aderência à instrução (0-5): A e B
3. Clareza/utilidade (0-5): A e B
4. Vencedor: A, B ou EMPATE
5. Justificativa breve

Formato:
Correção factual - A: X, B: Y
Aderência - A: X, B: Y
Clareza - A: X, B: Y
Vencedor: A/B/EMPATE
Justificativa: ..."""

In [37]:
results = []
for entry in tqdm(test_with_responses, desc="LLM-as-a-Judge"):
    prompt = JUDGE_PROMPT.format(
        instruction=entry["instruction"],
        input_text=entry.get("input", "(vazio)"),
        response_a=entry["response_base"],
        response_b=entry["response_finetuned"],
        reference=entry["output"]
    )
    judge_out = query_ollama(prompt)
    winner = "EMPATE"
    if "Vencedor: A" in judge_out or "vencedor: A" in judge_out.lower():
        winner = "A"
    elif "Vencedor: B" in judge_out or "vencedor: B" in judge_out.lower():
        winner = "B"
    results.append({"entry": entry, "judge_output": judge_out, "winner": winner})

wins_finetuned = sum(1 for r in results if r["winner"] == "B")
wins_base = sum(1 for r in results if r["winner"] == "A")
ties = sum(1 for r in results if r["winner"] == "EMPATE")
print(f"\n=== Resultados LLM-as-a-Judge (n={len(results)}) ===")
print(f"Vitórias fine-tuned (B): {wins_finetuned} ({100*wins_finetuned/len(results):.1f}%)")
print(f"Vitórias base (A): {wins_base} ({100*wins_base/len(results):.1f}%)")
print(f"Empates: {ties} ({100*ties/len(results):.1f}%)")

LLM-as-a-Judge: 100%|██████████| 2/2 [00:00<00:00, 253.46it/s]


=== Resultados LLM-as-a-Judge (n=2) ===
Vitórias fine-tuned (B): 0 (0.0%)
Vitórias base (A): 0 (0.0%)
Empates: 2 (100.0%)


In [38]:
print("--- Exemplo onde fine-tuned venceu ---")
for r in results:
    if r["winner"] == "B":
        e = r["entry"]
        print("Instrução:", e["instruction"][:80], "...")
        print("Base:", e["response_base"][:100])
        print("Fine-tuned:", e["response_finetuned"][:100])
        print("Esperado:", e["output"])
        print("Justificativa:", r["judge_output"][-200:] if len(r["judge_output"])>200 else r["judge_output"])
        break

print("\n--- Exemplo onde base venceu ---")
for r in results:
    if r["winner"] == "A":
        e = r["entry"]
        print("Instrução:", e["instruction"][:80], "...")
        print("Base:", e["response_base"][:100])
        print("Fine-tuned:", e["response_finetuned"][:100])
        break

--- Exemplo onde fine-tuned venceu ---

--- Exemplo onde base venceu ---
